In [ ]:
#Imports to make the Federated Learning Simulation
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import flwr as fl

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import os

In [3]:
# Dataset values
PATH_DATASET = "/content/drive/MyDrive/federated_learning/dataset2.csv"
TRAIN_RATIO = 0.8
INPUT_QUANTITY = 8
RANDOM_STATE = 42

# Federated Learning Dataset Distribution
ALPHA_DIRICHLET = 0.1

# Neural Network defined values
LOCAL_EPOCHS = 7
BATCH_SIZE = 32
LEARNING_RATE = 0.001

# Federated Learning defined values
NUMBER_HOSPITALS = 10
NUMBER_ROUNDS = 2

In [4]:
def loadData():

    df = pd.read_csv(PATH_DATASET)
    df = df.dropna()

    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

    df["activityID"] = df["activityID"].astype("category")
    categoriesLabel = df['activityID'].cat.categories
    df['activityID'] = df['activityID'].cat.codes

    numClass = len(categoriesLabel)

    x = df[[
        "heart_rate",
        "hand temperature (°C)",
        "hand acceleration X ±16g",
        "hand acceleration Y ±16g",
        "hand acceleration Z ±16g",
        "hand gyroscope X",
        "hand gyroscope Y",
        "hand gyroscope Z"
    ]].to_numpy().astype('float32')

    y = df["activityID"].to_numpy()

    return x, y, numClass

In [5]:
def partitionDataByDirichlet(x, y, numClass):

    np.random.seed(RANDOM_STATE)

    class_indices = [np.where(y == i)[0] for i in range(numClass)]
    client_indices = [[] for _ in range(NUMBER_HOSPITALS)]

    for c in range(numClass):
        indices = class_indices[c]
        np.random.shuffle(indices)

        proportions = np.random.dirichlet(ALPHA_DIRICHLET * np.ones(NUMBER_HOSPITALS))
        proportions = np.maximum(proportions, 1e-6)
        proportions = proportions / proportions.sum()

        proportions = (np.cumsum(proportions) * len(indices)).astype(int)[:-1]
        split_indices = np.split(indices, proportions)

        for client_id, idx in enumerate(split_indices):
            client_indices[client_id].extend(idx)

    client_data = []

    for i in range(NUMBER_HOSPITALS):
        idx = client_indices[i]

        if len(idx) == 0:
            client_data.append((np.array([]), np.array([])))
        else:
            client_data.append((x[idx], y[idx]))

    return client_data

In [6]:
def splitData(clientData):

    np.random.seed(RANDOM_STATE)

    split_data = []

    for x, y in clientData:
        n = len(x)

        if n == 0:
            split_data.append((x, y, x, y))
            continue

        idx = np.random.permutation(n)

        split = max(1, int(n * TRAIN_RATIO))

        train_idx = idx[:split]
        test_idx = idx[split:]

        x_train, y_train = x[train_idx], y[train_idx]
        x_test, y_test = x[test_idx], y[test_idx]

        split_data.append((x_train, y_train, x_test, y_test))

    return split_data

In [7]:
def create_model(numClass):

    model = keras.Sequential([
        layers.Input(shape=(INPUT_QUANTITY,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(numClass, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["sparse_categorical_accuracy"]
    )

    return model

In [8]:
def federated_average(weights, sizes):
    new_weights = []

    for layer_weights in zip(*weights):
        new_weights.append(
            np.sum([w * size for w, size in zip(layer_weights, sizes)], axis=0)
            / np.sum(sizes)
        )

    return new_weights

In [9]:
x, y, numClass = loadData()
partitions = partitionDataByDirichlet(x, y, numClass)
clients_data = splitData(partitions)

global_model = create_model(numClass)
global_weights = global_model.get_weights()

for round_num in range(NUMBER_ROUNDS):

    print(f"\n=== ROUND {round_num+1} ===")

    client_weights = []
    client_sizes = []

    client_metrics = []

    for client_id, (x_train, y_train, x_test, y_test) in enumerate(clients_data):

        if len(x_train) == 0:
            continue

        tf.keras.backend.clear_session()

        local_model = create_model(numClass)
        local_model.set_weights(global_weights)

        local_model.fit(
            x_train,
            y_train,
            epochs=LOCAL_EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=0
        )

        client_weights.append(local_model.get_weights())
        client_sizes.append(len(x_train))

        if len(x_test) > 0:
            y_pred = local_model.predict(x_test, verbose=0)
            y_pred = np.argmax(y_pred, axis=1)

            acc = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
            recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
            f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

            print(f"[Client {client_id}] Acc: {acc:.4f}")

            client_metrics.append((len(x_test), {
                "accuracy": acc,
                "precision": precision,
                "recall": recall,
                "f1": f1
            }))

    global_weights = federated_average(client_weights, client_sizes)
    global_model.set_weights(global_weights)

    if len(client_metrics) > 0:

        results = {}
        keys = client_metrics[0][1].keys()
        total = sum(n for n, _ in client_metrics)

        for key in keys:
            mean = sum(n * m[key] for n, m in client_metrics) / total
            results[key] = mean

            var = sum(n * ((m[key] - mean) ** 2) for n, m in client_metrics) / total
            results[f"{key}_var"] = var

        print("\n[GLOBAL METRICS]")
        for k, v in results.items():
            print(f"{k}: {v:.4f}")


=== ROUND 1 ===
[Client 0] Acc: 0.9209
[Client 1] Acc: 0.9647
[Client 2] Acc: 0.8502
[Client 3] Acc: 0.6978
[Client 4] Acc: 0.8563
[Client 5] Acc: 0.7829
[Client 6] Acc: 0.9446
[Client 7] Acc: 0.9453
[Client 8] Acc: 0.9640
[Client 9] Acc: 0.9435

[GLOBAL METRICS]
accuracy: 0.8973
accuracy_var: 0.0035
precision: 0.5299
precision_var: 0.0093
recall: 0.3800
recall_var: 0.0054
f1: 0.3980
f1_var: 0.0059

=== ROUND 2 ===
[Client 0] Acc: 0.9192
[Client 1] Acc: 0.9700
[Client 2] Acc: 0.8537
[Client 3] Acc: 0.7389
[Client 4] Acc: 0.8713
[Client 5] Acc: 0.8304
[Client 6] Acc: 0.9474
[Client 7] Acc: 0.9544
[Client 8] Acc: 0.9664
[Client 9] Acc: 0.9561

[GLOBAL METRICS]
accuracy: 0.9046
accuracy_var: 0.0029
precision: 0.5400
precision_var: 0.0072
recall: 0.4058
recall_var: 0.0054
f1: 0.4313
f1_var: 0.0060
